In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/2025-sep-dl-gen-ai-project/sample_submission.csv
/kaggle/input/2025-sep-dl-gen-ai-project/train.csv
/kaggle/input/2025-sep-dl-gen-ai-project/test.csv


In [ ]:
# K0 — Setup + W&B login
import os, random, numpy as np, torch, transformers, inspect
from pathlib import Path
import pandas as pd

# Quiet logs
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Repro
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| transformers:", transformers.__version__)

# Data paths
DATA_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")
TRAIN = DATA_DIR / "train.csv"
TEST  = DATA_DIR / "test.csv"
SAMPLE_SUB = DATA_DIR / "sample_submission.csv"
for p in [TRAIN, TEST, SAMPLE_SUB]:
    assert p.exists(), f"Missing: {p}"

# Labels/cols
ID_COL, TEXT_COL = "id", "text"
LABELS = ["anger","fear","joy","sadness","surprise"]
id2label = {i:l for i,l in enumerate(LABELS)}
label2id = {l:i for i,l in enumerate(LABELS)}

# Hyperparams
MODEL_NAME   = "roberta-base"
MAX_LEN      = 160
BATCH_SIZE   = 16  # lower to 12/8 if OOM
EPOCHS       = 4
LR           = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06

# ---- W&B login via Kaggle Secret ----
import wandb
try:
    from kaggle_secrets import UserSecretsClient
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
    assert key and len(key) > 25, "WANDB_API_KEY missing/invalid in Kaggle Secrets"
    os.environ["WANDB_API_KEY"] = key
    os.environ["WANDB_START_METHOD"] = "thread"
    os.environ["WANDB_INIT_TIMEOUT"] = "60"
    wandb.login(key=key, relogin=True)
    run = wandb.init(
        project="kaggle-emotions",
        name=f"roberta_f1_tuned_{SEED}",
        config=dict(
            model=MODEL_NAME, max_len=MAX_LEN, epochs=EPOCHS,
            batch_size=BATCH_SIZE, lr=LR, weight_decay=WEIGHT_DECAY, warmup_ratio=WARMUP_RATIO
        ),
        settings=wandb.Settings(start_method="thread", init_timeout=60),
        reinit=True
    )
    print("W&B URL:", run.url)
except Exception as e:
    raise RuntimeError(f"W&B init failed: {e}")


Device: cuda | transformers: 4.52.4


wandb: WARNING `start_method` is deprecated and will be removed in a future version of wandb. This setting is currently non-functional and safely ignored.


In [ ]:
# --- FIXED: rebuild datasets if needed, then build Trainer correctly and train ---

import inspect, re, torch, pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.nn import BCEWithLogitsLoss
from torch.utils.data import Dataset

# ===== Ensure tokenizer =====
if "tokenizer" not in globals() or isinstance(tokenizer, str):
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# ===== Ensure train_df/test_df =====
if "train_df" not in globals() or "test_df" not in globals():
    DATA_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")
    TRAIN = DATA_DIR / "train.csv"
    TEST  = DATA_DIR / "test.csv"
    train_df = pd.read_csv(TRAIN)
    test_df  = pd.read_csv(TEST)
    def clean_text(s):
        if not isinstance(s, str): s = "" if pd.isna(s) else str(s)
        s = s.lower(); s = re.sub(r"\s+", " ", s).strip()
        return s
    train_df["text"] = train_df["text"].apply(clean_text)
    test_df["text"]  = test_df["text"].apply(clean_text)

# ===== Torch datasets (if missing) =====
class EmotionTrainDS(Dataset):
    def __init__(self, df, text_col, labels, tok, max_len):
        self.df=df.reset_index(drop=True); self.text_col=text_col; self.labels=labels
        self.tok=tok; self.max_len=max_len
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        text = " ".join(str(self.df.loc[idx, self.text_col]).split())
        enc  = self.tok(text, truncation=True, padding="max_length",
                        max_length=self.max_len, return_tensors="pt")
        item = {k: v.squeeze(0) for k,v in enc.items()}
        y    = torch.tensor(self.df.loc[idx, self.labels].values.astype("float32"))
        item["labels"] = y
        return item

if "ds_train" not in globals() or "ds_val" not in globals():
    LABELS = ["anger","fear","joy","sadness","surprise"]
    label_counts = train_df[LABELS].sum(axis=1).clip(upper=len(LABELS))
    tr_df, va_df = train_test_split(train_df, test_size=0.2, random_state=SEED, stratify=label_counts)
    ds_train = EmotionTrainDS(tr_df, "text", LABELS, tokenizer, MAX_LEN)
    ds_val   = EmotionTrainDS(va_df,  "text", LABELS, tokenizer, MAX_LEN)

# ===== Ensure model object (not a string) =====
if ("model" not in globals()) or isinstance(model, str):
    id2label = {i:l for i,l in enumerate(LABELS)}
    label2id = {l:i for i,l in enumerate(LABELS)}
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(LABELS),
        problem_type="multi_label_classification",
        id2label=id2label, label2id=label2id
    )
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False

# ===== Ensure pos_weight tensor =====
if "pos_weight" not in globals():
    y_all = train_df[LABELS].astype(int).values
    pos = y_all.sum(axis=0); neg = y_all.shape[0] - pos
    pos_weight = torch.tensor((neg / (pos.clip(min=1)))).float()

# ===== Version-safe TrainingArguments =====
ta_kwargs = dict(
    output_dir="/kaggle/working/roberta_mlc",
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    num_train_epochs=EPOCHS,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=100,
    warmup_ratio=WARMUP_RATIO,
    report_to=["wandb"],
)
if "evaluation_strategy" in inspect.signature(TrainingArguments.__init__).parameters:
    ta_kwargs["evaluation_strategy"] = "epoch"
else:
    ta_kwargs["eval_strategy"] = "epoch"

training_args = TrainingArguments(**ta_kwargs)

# ===== v5-safe Trainer =====
class PosWeightTrainer(Trainer):
    def __init__(self, *args, pos_weight=None, **kwargs):
        self.pos_weight = pos_weight
        super().__init__(*args, **kwargs)
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels_t = inputs.pop("labels")
        outputs  = model(**inputs)
        logits   = outputs.logits
        if not torch.is_floating_point(labels_t):
            labels_t = labels_t.float()
        loss_fn = BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
        loss = loss_fn(logits, labels_t.to(logits.device))
        return (loss, outputs) if return_outputs else loss

trainer_kwargs = dict(
    model=model,
    args=training_args,
    train_dataset=ds_train,   # ✅ correct
    eval_dataset=ds_val,      # ✅ correct
    compute_metrics=lambda _: {},
    pos_weight=pos_weight,    # ✅ tensor, NOT weight decay
)
if "processing_class" in inspect.signature(Trainer.__init__).parameters:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = PosWeightTrainer(**trainer_kwargs)

print(">> Training …")
trainer.train()
print(">> Training complete.")


In [ ]:
# K2 (robust): rebuild ds_test if missing → tune thresholds → predict → save
import numpy as np, pandas as pd, torch, json, re
from pathlib import Path
from sklearn.metrics import f1_score, classification_report

# ---- Required from K1: trainer + ds_val + va_df ----
assert "trainer" in globals(), "Missing trainer. Run K1 first."
assert "ds_val" in globals(), "Missing ds_val. Run K1 first."
assert "va_df" in globals(), "Missing va_df. Run K1 first."

# ---- Ensure shared constants ----
LABELS   = globals().get("LABELS", ["anger","fear","joy","sadness","surprise"])
ID_COL   = globals().get("ID_COL", "id")
TEXT_COL = globals().get("TEXT_COL", "text")
MAX_LEN  = globals().get("MAX_LEN", 160)

# ---- Ensure tokenizer ----
from transformers import AutoTokenizer
if "tokenizer" not in globals():
    MODEL_NAME = globals().get("MODEL_NAME", "roberta-base")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# ---- Ensure test_df present; else load + clean from competition input ----
if "test_df" not in globals():
    COMP_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")
    TEST_csv = COMP_DIR / "test.csv"
    assert TEST_csv.exists(), "test.csv not found. Add competition dataset as input."
    test_df = pd.read_csv(TEST_csv)

# Ensure clean_text exists and text is cleaned (match training)
if "clean_text" not in globals():
    def clean_text(s):
        if not isinstance(s, str):
            s = "" if pd.isna(s) else str(s)
        s = s.lower()
        s = re.sub(r"\s+", " ", s).strip()
        return s
test_df[TEXT_COL] = test_df[TEXT_COL].apply(clean_text)

# ---- Rebuild ds_test if missing ----
from torch.utils.data import Dataset

if "EmotionTestDS" not in globals():
    class EmotionTestDS(Dataset):
        def __init__(self, df, text_col, tok, max_len):
            self.df=df.reset_index(drop=True); self.text_col=text_col
            self.tok=tok; self.max_len=max_len
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            text = " ".join(str(self.df.loc[idx, self.text_col]).split())
            enc  = self.tok(text, truncation=True, padding="max_length",
                            max_length=self.max_len, return_tensors="pt")
            return {k: v.squeeze(0) for k,v in enc.items()}

if "ds_test" not in globals():
    ds_test = EmotionTestDS(test_df, TEXT_COL, tokenizer, MAX_LEN)

# ================== Threshold tuning on validation ==================
val_pred   = trainer.predict(ds_val)
val_logits = val_pred.predictions
val_probs  = torch.sigmoid(torch.tensor(val_logits)).numpy()
y_val      = va_df[LABELS].astype(int).values

grid = np.linspace(0.2, 0.8, 25)
best_th, per_f1 = [], []
for i, lab in enumerate(LABELS):
    best_f, best_t = -1.0, 0.5
    p = val_probs[:, i]
    for t in grid:
        yhat = (p >= t).astype(int)
        f1 = f1_score(y_val[:, i], yhat, zero_division=0)
        if f1 > best_f:
            best_f, best_t = f1, t
    best_th.append(float(best_t)); per_f1.append(float(best_f))

val_pred_bin = (val_probs >= np.array(best_th)).astype(int)
macro_f1 = f1_score(y_val, val_pred_bin, average="macro", zero_division=0)
print("Val Macro F1:", round(macro_f1, 5))
print("Per-label F1:", dict(zip(LABELS, [round(x,4) for x in per_f1])))
print(classification_report(y_val, val_pred_bin, target_names=LABELS, zero_division=0))

# ================== Predict test & save submission ==================
test_pred   = trainer.predict(ds_test)
test_logits = test_pred.predictions
test_probs  = torch.sigmoid(torch.tensor(test_logits)).numpy()

sub = pd.DataFrame({ID_COL: test_df[ID_COL].values})
for i, lab in enumerate(LABELS):
    sub[lab] = (test_probs[:, i] >= best_th[i]).astype(int)
sub = sub[[ID_COL] + LABELS]

out_path = Path("/kaggle/working/submission.csv")
sub.to_csv(out_path, index=False)
print("Saved:", out_path)

# ================== (Optional) save artifacts & log to W&B ==================
import json as _json, pathlib, wandb
art_dir = pathlib.Path("/kaggle/working/artifacts")
(art_dir / "hf_model").mkdir(parents=True, exist_ok=True)
(art_dir / "tokenizer").mkdir(parents=True, exist_ok=True)
trainer.model.save_pretrained(art_dir / "hf_model")
(getattr(trainer, "tokenizer", tokenizer)).save_pretrained(art_dir / "tokenizer")
with open(art_dir / "thresholds.json","w") as f:
    _json.dump(dict(zip(LABELS, best_th)), f, indent=2)
print("Artifacts under:", art_dir)

if wandb.run is not None:
    wandb.run.summary["val_macro_f1"] = float(macro_f1)
    for i, lab in enumerate(LABELS):
        wandb.run.summary[f"val_f1_{lab}"] = float(per_f1[i])
    wandb.run.summary["thresholds"] = {lab: float(t) for lab, t in zip(LABELS, best_th)}
    wandb.save(str(out_path))
    for p in (art_dir / "hf_model").glob("*"): wandb.save(str(p))
    for p in (art_dir / "tokenizer").glob("*"): wandb.save(str(p))
    wandb.save(str(art_dir / "thresholds.json"))
    # wandb.finish()  # optional


In [ ]:
# K2.1 — Validate & fix submission.csv for Kaggle

import pandas as pd, numpy as np
from pathlib import Path

DATA_DIR = Path("/kaggle/input/2025-sep-dl-gen-ai-project")
TEST = DATA_DIR / "test.csv"
SAMPLE_SUB = DATA_DIR / "sample_submission.csv"
SUB_OUT = Path("/kaggle/working/submission.csv")

# 1) Load expected references
test_df = pd.read_csv(TEST)
sample  = pd.read_csv(SAMPLE_SUB)

# 2) Load your current submission (if exists)
assert SUB_OUT.exists(), "submission.csv not found at /kaggle/working/. Run K2 first."
sub = pd.read_csv(SUB_OUT)

# 3) Hard requirements
expected_cols = ["id","anger","fear","joy","sadness","surprise"]

# a) Columns exact & ordered
missing = [c for c in expected_cols if c not in sub.columns]
extra   = [c for c in sub.columns if c not in expected_cols]
if missing or extra:
    print("Fixing columns. Missing:", missing, " Extra:", extra)
    # If the shape looks right, try to realign by sample
    sub = sub[[c for c in expected_cols if c in sub.columns]]
    for c in expected_cols:
        if c not in sub.columns:
            sub[c] = 0
    sub = sub[expected_cols]

# b) Row count must match test
if len(sub) != len(test_df):
    print(f"Row count mismatch: sub={len(sub)} test={len(test_df)}. Rebuilding by join on id…")
    # If your sub has probs indexed differently, rebuild using sample ids
    # Keep existing predictions if present; otherwise fill 0
    sub = sample.copy()
    pred = pd.read_csv(SUB_OUT)
    if "id" in pred.columns:
        pred = pred.set_index("id")
        for c in expected_cols[1:]:
            if c in pred.columns:
                sub[c] = sub["id"].map(pred[c]).fillna(0)
    # else leave zeros

# c) Ensure id types & order match sample (some comps require exact order)
sub = sub.merge(sample[["id"]], on="id", how="right")  # enforce id set & order
sub = sub[expected_cols]

# d) Ensure binary 0/1 ints and no NaNs
for c in expected_cols[1:]:
    sub[c] = sub[c].clip(0,1).fillna(0).astype(int)

# e) Sanity
assert list(sub.columns) == expected_cols, "Column names/order incorrect."
assert len(sub) == len(test_df), "Row count must equal test size."
vals = set(np.unique(sub[expected_cols[1:]].values))
assert vals <= {0,1}, f"Found non-binary predictions: {vals}"

# 4) Save clean file
sub.to_csv(SUB_OUT, index=False)
print("✅ submission.csv validated & saved at:", SUB_OUT)
print(sub.head())
